# Data Extraction

All the raw data was downloaded as CSV manually from https://dados.turismo.gov.br/dataset/chegada-de-turistas-internacionais

In [1]:
import os

import pandas as pd
from tqdm.auto import tqdm

ROOT_PATH = os.path.dirname(os.getcwd())
DATA_FOLDER_PATH = os.path.join(ROOT_PATH, 'data')

## Data Import

Understanding how column names change over time

In [2]:
last_adopted_column_pattern = ''

for file in sorted(os.listdir(os.path.join(DATA_FOLDER_PATH, 'raw'))):
    if not file.endswith('.csv'):
        continue

    df = pd.read_csv(
        os.path.join(DATA_FOLDER_PATH, 'raw', file),
        encoding='latin-1',
        low_memory=False,
        sep=';',
    )

    if last_adopted_column_pattern != str(df.columns):
        last_adopted_column_pattern = str(df.columns)

        print(f'File: {file}')
        print(last_adopted_column_pattern)
        print('\n')

File: chegadas_1989.csv
Index(['Continente', 'Ordem continente', 'País', 'Ordem país', 'UF',
       'Ordem UF', 'Via de acesso', 'Ordem via de acesso', 'ano', 'Mês',
       'Ordem mês', 'Chegadas'],
      dtype='object')


File: chegadas_2000.csv
Index(['Continente', 'Ordem continente', 'País', 'Ordem país', 'UF',
       'Ordem UF', 'Via de acesso', 'Ordem via de acesso', 'Ano', 'Mês',
       'Ordem mês', 'Chegadas'],
      dtype='object')


File: chegadas_2015.csv
Index(['Continente', 'Ordem continente', 'País', 'Ordem país', 'UF',
       'Ordem UF', 'Via de acesso', 'Ordem via de acesso', 'ano', 'Mês',
       'Ordem mês', 'Chegadas'],
      dtype='object')


File: chegadas_2016.csv
Index(['Continente', 'cod continente', 'País', 'cod pais', 'UF', 'cod uf',
       'Via', 'cod via', 'ano', 'Mês', 'cod mes', 'Chegadas'],
      dtype='object')


File: chegadas_2025.csv
Index(['Via_de_acesso', 'UF', 'nome_pais_correto', 'mes', 'ano', 'Chegadas'], dtype='object')




In [3]:
COLUMN_MAPPING = {
    'access_route': ['Via de acesso', 'Via', 'Via_de_acesso'],
    'state': ['UF'],
    'origin_country': ['País', 'nome_pais_correto'],
    'month': ['Mês', 'mes'],
    'year': ['ano', 'Ano'],
    'arrival_count': ['Chegadas'],
}

SELECTED_COLUMNS = list(COLUMN_MAPPING.keys())


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {
        variant: target
        for target, variants in COLUMN_MAPPING.items()
        for variant in variants
        if variant in df.columns
    }
    return df.rename(columns=rename_map)[SELECTED_COLUMNS]

Unifying the data

In [4]:
dfs = []

for arrival_file in tqdm(os.listdir(os.path.join(DATA_FOLDER_PATH, 'raw'))):
    if not arrival_file.endswith('.csv'):
        continue

    try:
        df = pd.read_csv(
            os.path.join(DATA_FOLDER_PATH, 'raw', arrival_file),
            encoding='utf-8',
            low_memory=False,
            sep=';',
        )

    except UnicodeDecodeError:
        df = pd.read_csv(
            os.path.join(DATA_FOLDER_PATH, 'raw', arrival_file),
            encoding='latin-1',
            low_memory=False,
            sep=';',
        )

    df = standardize_columns(df)
    dfs.append(df)


df = pd.concat(dfs).reset_index(drop=True)

df.head()

  0%|          | 0/38 [00:00<?, ?it/s]

,access_route,state,origin_country,month,year,arrival_count
0,Aérea,Amazonas,África do Sul,janeiro,1989,9.0
1,Aérea,Amazonas,Angola,janeiro,1989,0.0
2,Aérea,Amazonas,Nigéria,janeiro,1989,0.0
3,Aérea,Amazonas,Outros países,janeiro,1989,0.0
4,Aérea,Amazonas,Costa Rica,janeiro,1989,6.0


Exporting

In [5]:
df.to_parquet(
    os.path.join(DATA_FOLDER_PATH, 'raw', 'raw_arrivals.parquet'), index=False
)